In [ ]:
"""Complete training pipeline for defect detection models."""

import os
import sys
from pathlib import Path

from abbvisionsystem.training_pipeline.data_manager import organize_enhanced_dataset, prepare_enhanced_yolo_dataset, generate_synthetic_defects
from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector, create_multi_object_test_images
from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel

In [ ]:
def run_complete_pipeline(
    source_data_dir: str,
    use_yolo: bool = True,
    use_classification: bool = True,
    train_yolo_epochs: int = 100,
    train_classification_epochs: int = 50,
    dataset_strategy: str = "mixed",  # New parameter for enhanced functionality
    balance_classes: bool = True,
    multi_object_scenes: int = 300
):
    """Run complete training pipeline for both YOLO and classification models with enhanced data handling."""
    
    print("🚀 Starting Enhanced Defect Detection Training Pipeline")
    print("=" * 60)
    print(f"📊 Dataset Strategy: {dataset_strategy}")
    print(f"⚖️  Class Balancing: {balance_classes}")
    print(f"🎬 Multi-object Scenes: {multi_object_scenes}")
    
    # Step 1: Organize enhanced dataset
    print("\n📁 Step 1: Organizing enhanced dataset...")
    classification_dataset = "training_data/enhanced_defect_detection_dataset"
    organize_enhanced_dataset(
        source_data_dir=source_data_dir,
        output_dir=classification_dataset,
        strategy=dataset_strategy,  # "cropped_only", "background_only", "mixed"
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        balance_classes=balance_classes
    )
    
    # Step 2: Prepare enhanced YOLO dataset
    print("\n🎯 Step 2: Preparing enhanced YOLO dataset...")
    yolo_dataset_yaml = prepare_enhanced_yolo_dataset(
        source_data_dir=source_data_dir,
        classification_dataset_dir=classification_dataset,
        output_dir="training_data/enhanced_yolo_dataset",
        multi_object_scenes=multi_object_scenes,
        use_legacy=False  # Use enhanced functionality
    )
    
    # Step 3: Create multi-object test images
    print("\n🖼️ Step 3: Creating multi-object test images...")
    create_multi_object_test_images(
        f"{classification_dataset}/test",
        "multi_object_test",
        images_per_composition=50  # More test images
    )
    
    results = {}
    
    # Step 4: Train YOLO model (recommended for your use case)
    if use_yolo:
        print("\n🤖 Step 4: Training Enhanced YOLO model...")
        yolo_detector = YOLODefectDetector()
        
        # Load appropriate model
        model_files = ["yolov8s.pt", "yolo11s.pt", "yolov8n.pt"]
        model_loaded = False
        
        for model_file in model_files:
            if yolo_detector.load_model(model_file):
                print(f"✅ Loaded model: {model_file}")
                model_loaded = True
                break
        
        if not model_loaded:
            print("❌ Failed to load any YOLO model. Skipping YOLO training.")
            results['yolo'] = None
        else:
            try:
                print(f"📈 Training with enhanced dataset containing:")
                print(f"   - Strategy: {dataset_strategy}")
                print(f"   - Multi-object scenes: {multi_object_scenes}")
                print(f"   - Class balancing: {balance_classes}")
                
                best_yolo_weights = yolo_detector.train(
                    dataset_yaml=yolo_dataset_yaml,
                    epochs=train_yolo_epochs,
                    imgsz=640,
                    batch=16,
                    project='trained_models',
                    name='enhanced_yolo_defect_detector'
                )
                
                # Evaluate on your "both" dataset
                print("\n📊 Evaluating YOLO model on real multi-object images...")
                yolo_results = evaluate_on_both_dataset(yolo_detector, f"{source_data_dir}/both")
                results['yolo'] = yolo_results
                
            except Exception as e:
                print(f"❌ YOLO training failed: {e}")
                print("💡 Possible solutions:")
                print("   - Try reducing batch size (batch=8 or batch=4)")
                print("   - Reduce image size (imgsz=320)")
                print("   - Ensure sufficient disk space")
                print("   - Check GPU memory (training will use CPU as fallback)")
                results['yolo'] = None
    
    # Step 5: Train classification model (for comparison)
    if use_classification:
        print("\n🧠 Step 5: Training Enhanced ResNet50V2 classification model...")
        classifier = DefectClassificationModel()
        classifier.build_model()
        
        try:
            # Prepare data with enhanced dataset
            train_gen, val_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/train",
                f"{classification_dataset}/validation"
            )
            
            print(f"📊 Training data prepared:")
            print(f"   Training samples: {train_gen.samples}")
            print(f"   Validation samples: {val_gen.samples}")
            
            # Train
            classifier.train(
                train_gen, val_gen,
                epochs=train_classification_epochs,
                model_name="enhanced_resnet_defect_classifier"
            )
            
            # Evaluate
            test_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/test",
                f"{classification_dataset}/test"
            )[1]  # Use validation generator (no augmentation)
            
            classification_results = classifier.evaluate(test_gen)
            results['classification'] = classification_results
            
            # Save model
            classifier.save_model("enhanced_resnet_defect_classifier")
            
            print(f"Enhanced Classification Results:")
            print(f"  Test Accuracy: {classification_results['test_accuracy']:.4f}")
            print(f"  Test Precision: {classification_results['test_precision']:.4f}")
            print(f"  Test Recall: {classification_results['test_recall']:.4f}")
            
        except Exception as e:
            print(f"❌ Classification training failed: {e}")
            results['classification'] = None
    
    # Step 6: Enhanced model comparison
    print("\n📈 Step 6: Enhanced Model Comparison Summary")
    print("=" * 50)
    
    if results.get('yolo') and results.get('classification'):
        print("🏆 Model Performance Comparison (Enhanced Dataset):")
        print(f"{'Metric':<15} {'YOLO':<12} {'ResNet50V2':<12} {'Best':<8}")
        print("-" * 47)
        
        # Compare accuracy
        yolo_acc = results['yolo']['accuracy']
        class_acc = results['classification']['test_accuracy']
        best_acc = "YOLO" if yolo_acc > class_acc else "ResNet50V2"
        print(f"{'Accuracy':<15} {yolo_acc:<12.4f} {class_acc:<12.4f} {best_acc:<8}")
        
        # Compare precision
        yolo_prec = results['yolo']['precision']
        class_prec = results['classification']['test_precision']
        best_prec = "YOLO" if yolo_prec > class_prec else "ResNet50V2"
        print(f"{'Precision':<15} {yolo_prec:<12.4f} {class_prec:<12.4f} {best_prec:<8}")
        
        # Compare recall
        yolo_recall = results['yolo']['recall']
        class_recall = results['classification']['test_recall']
        best_recall = "YOLO" if yolo_recall > class_recall else "ResNet50V2"
        print(f"{'Recall':<15} {yolo_recall:<12.4f} {class_recall:<12.4f} {best_recall:<8}")
        
        # Calculate and compare F1 scores
        yolo_f1 = results['yolo']['f1_score']
        precision = results['classification']['test_precision']
        recall = results['classification']['test_recall']
        class_f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        best_f1 = "YOLO" if yolo_f1 > class_f1 else "ResNet50V2"
        print(f"{'F1 Score':<15} {yolo_f1:<12.4f} {class_f1:<12.4f} {best_f1:<8}")
        
        # Additional YOLO-specific metrics
        print(f"\n🎯 YOLO-Specific Metrics:")
        print(f"  Detection Rate: {results['yolo']['detection_rate']:.4f}")
        print(f"  Avg Detections/Image: {results['yolo']['avg_detections_per_image']:.2f}")
        print(f"  Total Detections: {results['yolo']['total_detections']}")
        
    elif results.get('yolo'):
        print("🎯 YOLO Model Results (Enhanced Dataset):")
        yolo_results = results['yolo']
        print(f"  Accuracy: {yolo_results['accuracy']:.4f}")
        print(f"  Precision: {yolo_results['precision']:.4f}")
        print(f"  Recall: {yolo_results['recall']:.4f}")
        print(f"  F1 Score: {yolo_results['f1_score']:.4f}")
        print(f"  Detection Rate: {yolo_results['detection_rate']:.4f}")
        
    elif results.get('classification'):
        print("🧠 Classification Model Results (Enhanced Dataset):")
        class_results = results['classification']
        print(f"  Accuracy: {class_results['test_accuracy']:.4f}")
        print(f"  Precision: {class_results['test_precision']:.4f}")
        print(f"  Recall: {class_results['test_recall']:.4f}")
    
    print("\n✅ Enhanced Pipeline completed successfully!")
    print("\n🎯 ENHANCED RECOMMENDATION FOR YOUR USE CASE:")
    print("With your rich dataset including background images and multiple categories,")
    print("the enhanced YOLO model should perform significantly better because:")
    print("  • 🌄 Trained on real backgrounds from your colorless datasets")
    print("  • 🔄 Handles deformed defects from defect_colorless_deform")
    print("  • 📝 Processes images without text from defect_colorless_nowords")
    print("  • 🎭 Creates realistic multi-object scenes")
    print("  • ⚖️  Balanced training data across all categories")
    print("  • 🎯 Optimized for real-world production scenarios")
    
    return results

In [ ]:
def evaluate_on_both_dataset(yolo_detector, both_images_dir):
    """Evaluate on your 'both' dataset with multiple objects."""
    if not os.path.exists(both_images_dir):
        print(f"⚠️  'both' dataset directory not found: {both_images_dir}")
        # Return default metrics structure to avoid comparison errors
        return {
            "total_images": 0,
            "images_with_detections": 0,
            "total_detections": 0,
            "avg_detections_per_image": 0.0,
            "confidence_scores": [],
            "detection_rate": 0.0,
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1_score": 0.0
        }
    
    results = {
        "total_images": 0,
        "images_with_detections": 0,
        "total_detections": 0,
        "avg_detections_per_image": 0.0,
        "confidence_scores": [],
        "detection_rate": 0.0,
        "accuracy": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "f1_score": 0.0
    }
    
    image_files = [f for f in os.listdir(both_images_dir) 
                   if f.endswith(('.jpg', '.jpeg', '.png', '.JPG'))]
    
    # Metrics tracking
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    
    for img_file in image_files:
        img_path = os.path.join(both_images_dir, img_file)
        
        try:
            detections = yolo_detector.predict(img_path, conf_threshold=0.25)
            
            results["total_images"] += 1
            num_detections = len(detections["boxes"])
            defect_detections = sum(1 for cls in detections["classes"] if cls == 1)
            
            if num_detections > 0:
                results["images_with_detections"] += 1
                results["total_detections"] += num_detections
                results["confidence_scores"].extend(detections["scores"])
            
            # For evaluation, assume images with "defect" in filename are defective
            # You may need to adjust this logic based on your actual labeling
            is_defective_image = "defect" in img_file.lower() or "bad" in img_file.lower()
            
            if is_defective_image and defect_detections > 0:
                true_positives += 1
            elif is_defective_image and defect_detections == 0:
                false_negatives += 1
            elif not is_defective_image and defect_detections > 0:
                false_positives += 1
            elif not is_defective_image and defect_detections == 0:
                true_negatives += 1
                
        except Exception as e:
            print(f"❌ Error processing {img_file}: {str(e)}")
            continue
    
    # Calculate metrics
    if results["total_images"] > 0:
        results["avg_detections_per_image"] = results["total_detections"] / results["total_images"]
        results["detection_rate"] = results["images_with_detections"] / results["total_images"]
        
        # Calculate classification metrics
        total_predictions = true_positives + false_positives + false_negatives + true_negatives
        if total_predictions > 0:
            results["accuracy"] = (true_positives + true_negatives) / total_predictions
        
        if true_positives + false_positives > 0:
            results["precision"] = true_positives / (true_positives + false_positives)
        
        if true_positives + false_negatives > 0:
            results["recall"] = true_positives / (true_positives + false_negatives)
        
        if results["precision"] + results["recall"] > 0:
            results["f1_score"] = 2 * (results["precision"] * results["recall"]) / (results["precision"] + results["recall"])
    
    print(f"📊 YOLO Evaluation Results on 'both' dataset:")
    print(f"   Total images: {results['total_images']}")
    print(f"   Images with detections: {results['images_with_detections']}")
    print(f"   Total detections: {results['total_detections']}")
    print(f"   Detection rate: {results['detection_rate']:.4f}")
    print(f"   Accuracy: {results['accuracy']:.4f}")
    print(f"   Precision: {results['precision']:.4f}")
    print(f"   Recall: {results['recall']:.4f}")
    print(f"   F1 Score: {results['f1_score']:.4f}")
    
    return results

In [ ]:
def test_pipeline_setup():
    """Test if all components are properly set up for enhanced functionality."""
    print("🔍 Testing enhanced pipeline setup...")
    
    try:
        from abbvisionsystem.training_pipeline.data_manager import organize_enhanced_dataset, prepare_enhanced_yolo_dataset
        print("✅ Enhanced data_manager imports successful")
    except ImportError as e:
        print(f"❌ Enhanced data_manager import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector
        print("✅ yolo_trainer import successful")
    except ImportError as e:
        print(f"❌ yolo_trainer import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel
        print("✅ resnet_trainer import successful")
    except ImportError as e:
        print(f"❌ resnet_trainer import failed: {e}")
        return False
    
    # Check if ultralytics is available for YOLO
    try:
        from ultralytics import YOLO
        print("✅ ultralytics available")
        
        # Test model loading
        yolo_detector = YOLODefectDetector()
        test_models = ["yolov8s.pt", "yolo11s.pt", "yolov8n.pt"]
        model_available = False
        
        for model in test_models:
            try:
                if yolo_detector.load_model(model):
                    print(f"✅ YOLO model {model} loadable")
                    model_available = True
                    break
            except:
                continue
        
        if not model_available:
            print("⚠️  No YOLO models could be loaded. Training may download models automatically.")
        
    except ImportError:
        print("❌ ultralytics not installed. Install with: pip install ultralytics")
        return False
    
    # Check if tensorflow is available
    try:
        import tensorflow as tf
        print(f"✅ tensorflow {tf.__version__} available")
    except ImportError:
        print("❌ tensorflow not installed")
        return False
    
    # Test enhanced data manager functionality
    try:
        from abbvisionsystem.training_pipeline.data_manager import EnhancedDataManager
        print("✅ EnhancedDataManager class available")
    except ImportError as e:
        print(f"❌ EnhancedDataManager import failed: {e}")
        return False
    
    print("✅ Enhanced pipeline setup test completed successfully!")
    print("🎯 Ready for enhanced training with rich dataset support!")
    return True


In [ ]:
if __name__ == "__main__":
    # First test the setup
    if not test_pipeline_setup():
        print("❌ Setup test failed. Please fix the issues above.")
        exit(1)
    
    # Run the enhanced pipeline
    source_dir = "data/choco-pie"  # Update this path
    
    if not os.path.exists(source_dir):
        print(f"Source directory {source_dir} not found!")
        print("Please update the source_dir variable to point to your data.")
        print("Expected enhanced structure:")
        print("data/choco-pie/")
        print("├── good/                    # Cropped good images")
        print("├── defect/                  # Cropped defect images")
        print("├── good_colorless/          # Good images with backgrounds")
        print("├── defect_colorless/        # Defect images with backgrounds")
        print("├── defect_colorless_deform/ # Deformed defects with backgrounds")
        print("├── defect_colorless_nowords/# Defects without text")
        print("└── both/                    # Mixed test images")
    else:
        # Enhanced data structure analysis
        print("🔍 Analyzing enhanced data structure...")
        
        # Check for enhanced directories
        enhanced_dirs = [
            "good", "defect", "good_colorless", "defect_colorless", 
            "defect_colorless_deform", "defect_colorless_nowords", "both"
        ]
        
        available_dirs = []
        for dir_name in enhanced_dirs:
            dir_path = os.path.join(source_dir, dir_name)
            if os.path.exists(dir_path):
                available_dirs.append(dir_name)
        
        print(f"📊 Available data directories: {', '.join(available_dirs)}")
        
        # Determine best strategy based on available data
        has_cropped = any(d in available_dirs for d in ["good", "defect"])
        has_background = any(d in available_dirs for d in ["good_colorless", "defect_colorless", 
                                                          "defect_colorless_deform", "defect_colorless_nowords"])
        
        if has_cropped and has_background:
            strategy = "mixed"
            print("🎯 Using MIXED strategy - leveraging all available data")
        elif has_background:
            strategy = "background_only" 
            print("🌄 Using BACKGROUND_ONLY strategy - using images with backgrounds")
        elif has_cropped:
            strategy = "cropped_only"
            print("📸 Using CROPPED_ONLY strategy - using cropped images")
        else:
            strategy = "legacy"
            print("⚠️  Falling back to LEGACY strategy - basic good/defect structure")
        
        # Run enhanced pipeline with optimal settings
        results = run_complete_pipeline(
            source_data_dir=source_dir,
            use_yolo=True,
            use_classification=True,
            train_yolo_epochs=100,  # Increased for better performance with rich data
            train_classification_epochs=50,
            dataset_strategy=strategy,  # Automatically determined strategy
            balance_classes=True,      # Balance classes for better training
            multi_object_scenes=400   # More scenes for better generalization
        )
        
        # Additional evaluation on enhanced test set
        if results.get('yolo') and os.path.exists(os.path.join(source_dir, "both")):
            print("\n🧪 Additional Enhanced Evaluation:")
            print("Testing on 'both' dataset with real multi-object scenarios...")
